<a href="https://colab.research.google.com/github/hirasaeed5122005-blip/Hand-gesture-recognition/blob/main/Project1_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install ultralytics gradio -q

from ultralytics import YOLO
import gradio as gr
from google.colab import files
import zipfile

print("Please upload your dataset ZIP file")
uploaded = files.upload() # Upload button will appear

# Save uploaded file
for name, data in uploaded.items():
    with open(name, 'wb') as f:
        f.write(data)
    print(f'{name} uploaded successfully')

# Extract dataset
zip_name = list(uploaded.keys())[0]
!unzip -q "{zip_name}" -d dataset
print("Dataset extracted to 'dataset' folder")

Please upload your dataset ZIP file


Saving Hand Gesture.v1i.yolov8.zip to Hand Gesture.v1i.yolov8 (1).zip
Hand Gesture.v1i.yolov8 (1).zip uploaded successfully
replace dataset/test/images/image_59_png.rf.cfab90abb7b84e81963b35df054a0aac.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Dataset extracted to 'dataset' folder


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(data='dataset/data.yaml', epochs=30, imgsz=640)
print("Training completed!")

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=None, optimize=False, opt

In [ ]:
from ultralytics import YOLO
import gradio as gr

best_model = YOLO('runs/detect/train-2/weights/best.pt')

def predict(image):
    results = best_model(image, conf=0.25)
    annotated_image = results[0].plot()

    if len(results[0].boxes) > 0:
        labels = list(set([best_model.names[int(box.cls)] for box in results[0].boxes]))
        confs = [float(box.conf) for box in results[0].boxes]
        max_conf = round(max(confs)*100, 1)
        gesture_name = labels[0] # pehla wala naam le lo

        result_text = f"✅ Hand Gesture Detected!\n\nGesture: {gesture_name}\nConfidence: {max_conf}%"
    else:
        result_text = "❌ No Hand Detected\n\nTry these gestures: C-Shape, Fist, L-Shape, Like, Peace, Stop"

    return annotated_image, result_text

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=["image", "text"],
    title="✋ Hand Gesture Detection using YOLOv8",
    description="Upload an image to detect hand gestures. Supported: C-Shape, Fist, L-Shape, Like, Peace, Stop"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c8ba6aab6c329a25ba.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
